In [ ]:
import numpy as np
import pickle as pk
import uproot as ur
import pandas as pd
import awkward_pandas as akpd

### This script will likely crash if you have too many other memory constraints on your computer.
### the root file, when first opened, has 2x16 channels which consume a large amount of memory.
### I recommend having nothing else running besides this terminal to avoid crashing.


# this script is only designed for HPGE data. I can make another one for
# the neutron monitor if there is a demand for it.

In [1]:
# change any paths / filenames / tree strings to suit your data

run_number = 70 
pathin = f"/path/to/rootfiles/" # directory for rootfiles to pre-process
pathout = f"/path/to/outputdirectory/forpklfiles/"       # directory for reduced .pkl files to be output

# check that this string matches your TTree's string
treestr = f"SSA{run_number}_HPGE"
                      
# check that this naming convention matches your root file
infile = f"root_data_SSA_0{run_number}.bin_tree.root"
infile = pathin+infile
outfile = f"SSA{run_number}.pkl"
outfile = pathout+outfile


# for HPGE data, detectors are indices 0,2,4,6
# the trigger_time at beam pickoff is index 0
columns = ['amplitude', # 16 channels
                'channel_time', # 16 channels
                'trigger_time' # 2 channels
          ]

hpge_indices = [0,2,4,6]
nmon_index = [14]
trigger_index = 0

empty_detectors = [] # empty columns for amplitude/channel_time
for i in range(16):
    if i not in hpge_indices:
        empty_detectors.append(i)

empty_trigger_time = [1]

amplitude_col = {i: f"amplitude[{i}]" for i in hpge_indices}
channel_time_col = {i: f'channel_time[{i}]' for i in hpge_indices}
trigger_time_col = {trigger_index: f'trigger_time[{trigger_index}]'}

column_strings = ['amplitude', 'channel_time', 'trigger_time']
column_mappings = [amplitude_col, channel_time_col, trigger_time_col]
drop_columns = [empty_detectors, empty_detectors, empty_trigger_time]

In [ ]:
dflist = []

# we open the file and save a dataframe for one set of columns at a time.
# first we save amplitudes, then channel times, then the trigger times.
# opening all of them at once will crash computer memeory.

for string, mapping, drop_col in zip(column_strings, column_mappings, drop_columns):
    with ur.open(infile) as f: 
        tempdf = f[treestr].arrays(string, library='np')[string]
        tempdf = pd.DataFrame(tempdf)
        tempdf.drop(columns=drop_col, inplace=True)
        tempdf.rename(mapper=mapping, axis=1, inplace=True)
        dflist.append(tempdf)
    print(f'finished {string} column')
pd.concat(dflist, axis=1).to_pickle(outfile) # concatenate amplitude, channel, and trigger
print(f'file created: {outfile}')